# Field-level inference of the cosmology at fixed initial conditions

Notebook 16 samples $(\Omega_c,\sigma_8)$ jointly with the $128^3$ white-noise initial-condition (IC) field behind two convergence maps. Its MCLMC chain reconstructs the IC and $\kappa$ well, but its 100 draws have an effective sample size of 3 to 5 in the cosmology. Here we fix the white-noise field $w$ and sample only $(\Omega_c,\sigma_8)$ with the No-U-Turn Sampler (NUTS), with the model, truth and data of notebook 16. The result is the conditional posterior $p(\Omega_c,\sigma_8\mid w,\ \mathrm{data})$, for one of two fields:
- `IC = "mean"`: the posterior mean of the white field over the 100 draws of notebook 16, each draw whitened with its own cosmology;
- `IC = "truth"`: the white field that generated the data, which tests the forward model and the likelihood.

The white field enters the model before the colouring with the linear power spectrum, so at fixed $w$ a change of $\sigma_8$ rescales every mode of the IC. The conditional posterior is therefore narrow, and its position depends on $w$. At the true field it recovers the truth. At the posterior mean, whose amplitude the average over correlated draws shrinks, the cosmology shifts to compensate. A grid of the log-posterior around the NUTS draws cross-checks each run; it needs the forward model only, 0.6 s per point on an A100.

Every NUTS step runs the forward model and its gradient, 8 s on an A100. On the grid the log-posterior fluctuates by about one unit between cosmologies $10^{-3}$ apart. These fluctuations limit the acceptance of NUTS steps, so the step-size adaptation targets a low acceptance (`TARGET_ACCEPT`). A run of `NUM_WARMUP` + `NUM_SAMPLES` × `BATCH_COUNT` NUTS iterations takes a few hours.

## 1. Environment

Everything runs in float64, with the CUDA spherical-harmonic transforms on a GPU.

In [ ]:
import os

os.environ["JAX_ENABLE_X64"] = "True"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["HF_DATASETS_OFFLINE"] = "1"

import threading
import time
from pathlib import Path

import datasets
import jax
import jax.numpy as jnp
import jax_cosmo as jc
import matplotlib.pyplot as plt
import numpy as np
import numpyro
import numpyro.distributions as dist
from jax._src import dispatch
from numpyro.handlers import condition, reparam, seed, trace
from numpyro.infer.reparam import Reparam
from numpyro.infer.util import potential_energy
from scipy.special import ndtri

import jax_fli as jfli
from jax_fli.data.nz import get_des_y3_nz_shear

datasets.disable_progress_bar()
jax.config.update("jax_enable_x64", True)
MAP2ALM = "jax_cuda" if jax.default_backend() == "gpu" else "jax"
print(jax.devices(), MAP2ALM)

## 2. Parameters

NUTS adapts its step size and a diagonal mass matrix over `NUM_WARMUP` iterations, then stores `NUM_SAMPLES` draws per batch for `BATCH_COUNT` batches. An iteration costs at most $2^{\texttt{MAX\_DOUBLINGS}}$ gradient evaluations, and successive draws are nearly independent. `GRID_POINTS` sets the number of points of the cross-check grid along each parameter. `ROOT` holds the run of notebook 16 and the runs of this notebook, in `mcmc_results/` next to this notebook.

In [ ]:
RUN_SAMPLING = False  # True runs NUTS and the grid on a GPU; False reads them from ROOT
IC = "mean"  # "mean": posterior-mean white field of notebook 16; "truth": the white field of the data
MESH = 128
INIT = "truth"  # notebook 16's run
NUM_WARMUP = 100  # NUTS warmup iterations
NUM_SAMPLES = 10  # draws per batch
BATCH_COUNT = 10
MAX_DOUBLINGS = 4  # at most 2**MAX_DOUBLINGS gradients per iteration
TARGET_ACCEPT = 0.3  # the log-posterior fluctuates by ~1 unit between neighbouring cosmologies
GRID_POINTS = 31  # grid points along Omega_c and along sigma8
ROOT = "mcmc_results"  # the runs, next to this notebook

## 3. Survey, resolution and model

The survey, the resolution and the model are those of notebook 16, sections 3 and 4.

In [ ]:
MAX_Z = 1.0
N_SHELLS, MIN_WIDTH, MAX_WIDTH, R_MIN = 22, 50.0, 150.0, 300.0  # Mpc/h
SEED = 0
cosmo = jc.Planck18()
box = tuple(float(x) for x in jfli.utils.compute_box_size_from_redshift(cosmo, MAX_Z, (0.5, 0.5, 0.5)))
NZ = [get_des_y3_nz_shear(gals_per_arcmin2=[30.0 / 4] * 4, zmax=MAX_Z)[i] for i in (1, 2)]
SIGMA_E = 0.30 / np.sqrt(2)
N_BINS = len(NZ)

# lensing efficiency of each bin on a grid of comoving distance, for ELL_MAX
chi = np.linspace(1.0, box[0] / 2, 2000)
z_src = np.linspace(1e-3, MAX_Z, 1000)
chi_src = np.asarray(jc.background.radial_comoving_distance(cosmo, jc.utils.z2a(jnp.asarray(z_src))))
a_chi = np.asarray(jc.background.a_of_chi(cosmo, jnp.asarray(chi)))
efficiency = []
for nz in NZ:
    n_z = np.asarray(nz(jnp.asarray(z_src)))
    geometry = np.clip(1 - chi[:, None] / chi_src, 0, None)
    efficiency.append(chi / a_chi * np.trapezoid(n_z * geometry, z_src, axis=1) / np.trapezoid(n_z, z_src))
chi_peak = min(chi[np.argmax(q)] for q in efficiency)

ELL_MAX = int(np.pi * MESH / box[0] * chi_peak)
ELL_TAPER = round(0.125 * ELL_MAX)
PAINT_NSIDE = int(2 ** np.ceil(np.log2(ELL_MAX / 2)))
NSIDE = 2 * PAINT_NSIDE
OUT = Path(ROOT) / f"FIELDLEVEL_MESH{MESH}_{INIT}"
CHAIN = OUT / "chain"
TRUTH = OUT / "truth"
FIXED = OUT / f"cosmology_fixed_ic_{IC}"
FIXED.mkdir(parents=True, exist_ok=True)

priors = {"Omega_c": jfli.infer.PreconditionnedUniform(0.1, 0.5), "sigma8": jfli.infer.PreconditionnedUniform(0.6, 1.0)}
config = jfli.ppl.Configurations(
    mesh_size=(MESH,) * 3,
    box_size=box,
    halo_size=(0, 0),
    field_sharding=None,
    sim_mode="lpt",
    lpt_order=2,
    paint_order="cic",
    gradient_order=4,
    laplace_fd=True,
    number_of_shells=N_SHELLS,
    shell_spacing="equal_vol",
    min_width=MIN_WIDTH,
    max_width=MAX_WIDTH,
    r_min=R_MIN,
    resolution_cut=True,
    geometry="spherical",
    scheme="bilinear",
    nside=NSIDE,
    paint_nside=PAINT_NSIDE,
    kernel_width_pixels=0.8,
    observer_position=(0.5, 0.5, 0.5),
    lensing_output="convergence",
    normalization="global",
    map2alm_method=MAP2ALM,
    min_redshift=0.001,
    max_redshift=MAX_Z,
    n_integrate=8,
    fiducial_cosmology=jc.Planck18,
    nz_shear=NZ,
    priors=priors,
    sigma_e=SIGMA_E,
    ell_max=ELL_MAX,
    ell_taper_width=ELL_TAPER,
    ell_min=2,
    log_observable=True,
)
model = jfli.ppl.full_field_probmodel(config)
print(f"MESH {MESH}^3, ELL_MAX {ELL_MAX}, NSIDE {NSIDE}, IC {IC!r}, output {FIXED}")

## 4. Truth, data and the joint chain

The truth and the data are those of notebook 16, regenerated from the same seed. The cosmology of its 100 joint draws serves as the reference.

In [ ]:
fiducial_base = {
    f"{name}_base": float(
        ndtri((float(getattr(cosmo, name)) - float(prior.low)) / (float(prior.high) - float(prior.low)))
    )
    for name, prior in priors.items()
}
white_truth = jax.random.normal(jax.random.PRNGKey(SEED), config.mesh_size)
truth = {k: jnp.asarray(v) for k, v in fiducial_base.items()} | {"initial_conditions": white_truth}
if RUN_SAMPLING:
    mock = trace(seed(condition(model, data=truth), SEED)).get_trace()
    data = {f"observable_{b}": mock[f"observable_{b}"]["value"] for b in range(N_BINS)}
ic_truth = jfli.io.Catalog.from_parquet(str(TRUTH / "ic_truth.parquet")).field[0]
joint = jfli.io.extract_catalog(
    set_name="joint MCLMC, IC sampled (notebook 16)",
    cosmo_keys=["Omega_c", "sigma8"],
    patterns=[str(CHAIN / "samples")],
    truth=jfli.io.Catalog(field=[ic_truth], cosmology=[cosmo]),
)
for name in ("Omega_c", "sigma8"):
    draws = np.asarray(joint.cosmo[name])
    print(f"joint chain: {name:8s} {draws.mean():.4f} +/- {draws.std():.4f}   truth {joint.truth_cosmo[name]:.4f}")

## 5. The fixed initial conditions

The draws of notebook 16 store the physical IC $\delta_i$, coloured with the cosmology $\theta_i$ of each draw. We undo the colouring with the inverse square root of the same power spectrum, $w_i=\mathcal F^{-1}\big[\mathcal F[\delta_i]/\sqrt{P(k;\theta_i)\,N/V}\big]$, through `interpolate_initial_conditions` with the reciprocal spectrum, and average the whitened draws. On the truth this inverse recovers the white field to $10^{-14}$. The fixed field is saved to `white_ic.parquet`.

In [ ]:
def whiten(delta, cosmo_draw):
    # inverse of interpolate_initial_conditions: divide the modes by sqrt(P(k) N/V) instead of multiplying
    k = jnp.logspace(-4, 1, 128)
    pk = jc.power.linear_matter_power(cosmo_draw, k)
    factor = MESH**3 / (box[0] * box[1] * box[2])

    def inverse_pk(x):
        return 1.0 / (jnp.interp(x, k, pk) * factor**2)

    return jfli.interpolate_initial_conditions(delta, config.mesh_size, config.box_size, pk_fn=inverse_pk).array.real


if RUN_SAMPLING:
    if IC == "truth":
        white_ic = white_truth
    else:
        white_ic, n_draws = jnp.zeros(config.mesh_size), 0
        for f in sorted((CHAIN / "samples" / "samples").glob("samples_*.parquet")):
            draws = jfli.io.Catalog.from_parquet(str(f))
            for field, cosmo_draw in zip(draws.field, draws.cosmology):
                white_ic, n_draws = white_ic + whiten(field.array, cosmo_draw), n_draws + 1
        white_ic = white_ic / n_draws
    jfli.io.Catalog(field=[ic_truth.replace(array=white_ic)], cosmology=[cosmo]).to_parquet(
        str(FIXED / "white_ic.parquet")
    )
white_ic = jfli.io.Catalog.from_parquet(str(FIXED / "white_ic.parquet")).field[0].array
print(
    f"fixed white field: std {float(white_ic.std()):.3f}, "
    f"correlation with the true white field {float(jnp.corrcoef(white_ic.ravel(), white_truth.ravel())[0, 1]):.3f}"
)

## 6. Posterior at fixed initial conditions

We condition the model on the data and on the fixed white field, so that only the two cosmological base variables remain. As in notebook 16, NUTS samples $z_k=\theta_k/s_k$, with $s_k$ the conditional width of each base variable at the truth, measured by a central difference of the gradient.

In [ ]:
class ScaleReparam(Reparam):
    # theta = scale * z: the sampler moves z, whose posterior width is ~1 when scale is the width of theta
    def __init__(self, scale):
        self.scale = scale

    def __call__(self, name, fn, obs):
        z = numpyro.sample(
            f"{name}_z", dist.TransformedDistribution(fn, dist.transforms.AffineTransform(0.0, 1.0 / self.scale))
        )
        return None, self.scale * z


if RUN_SAMPLING:
    posterior = condition(model, data=data | {"initial_conditions": white_ic})

    def potential(position):
        return potential_energy(posterior, (), {}, position)

    value_and_grad = jax.jit(jax.value_and_grad(potential))
    scales = {}
    for name in fiducial_base:
        grad_plus = value_and_grad(fiducial_base | {name: fiducial_base[name] + 1e-3})[1][name]
        grad_minus = value_and_grad(fiducial_base | {name: fiducial_base[name] - 1e-3})[1][name]
        scales[name] = float((2e-3 / (grad_plus - grad_minus)) ** 0.5)
    print("conditional widths of the base variables:", scales)
    sampled_posterior = reparam(posterior, config={name: ScaleReparam(s) for name, s in scales.items()})
    start = {f"{name}_z": jnp.asarray(fiducial_base[name] / s) for name, s in scales.items()}

## 7. Sampling

`jfli.infer.batched_sampling` runs the window adaptation of NUTS in chunks, then samples in batches, with the same checkpoints and resumption as in notebook 16. No IC is sampled, so `sample2catalog` writes the cosmology of each batch to `chain/samples/samples/cosmo_<batch>.npz` and the predicted $\kappa$ to `chain/samples/observable_fields/`. A background thread clears jax-cosmo's eager compilation cache every 5 minutes for the rest of the session, through the sampling and the grid of section 8, as in notebook 16.

In [ ]:
if RUN_SAMPLING:

    def clear_eager_cache():
        # the eager jit(scan) of jax-cosmo's host cache, one per cosmology: free them before the mapping limit
        while True:
            time.sleep(300)
            dispatch.xla_primitive_callable.cache_clear()

    threading.Thread(target=clear_eager_cache, daemon=True).start()
    jfli.infer.batched_sampling(
        sampled_posterior,
        path=str(FIXED / "chain"),
        rng_key=jax.random.PRNGKey(2),
        num_warmup=NUM_WARMUP,
        num_samples=NUM_SAMPLES,
        batch_count=BATCH_COUNT,
        sampler="NUTS",
        thinning=1,
        max_num_doublings=MAX_DOUBLINGS,
        target_accept=TARGET_ACCEPT,
        init_params=start,
        progress_bar=True,
        print_rate=1,
        save_callback=jfli.infer.sample2catalog(config),
        post_process=None,
    )
print((FIXED / "chain" / "metrics.md").read_text())
nuts = jfli.io.extract_cosmo_catalog(
    set_name=f"NUTS, IC fixed ({IC})",
    cosmo_keys=["Omega_c", "sigma8"],
    patterns=[str(FIXED / "chain" / "samples")],
    truth=cosmo,
)
for name in ("Omega_c", "sigma8"):
    draws = np.asarray(nuts.cosmo[name])
    print(f"NUTS: {name:8s} {draws.mean():.4f} +/- {draws.std():.4f}   truth {nuts.truth_cosmo[name]:.4f}")

## 8. Cross-check on a grid

We evaluate the log-posterior on a grid of `GRID_POINTS` × `GRID_POINTS` points that covers five standard deviations of the NUTS draws on each side (at least 0.01), with the forward model only. The density in $(\Omega_c,\sigma_8)$ is $e^{-U}$ times the Jacobian of the map from the base variables, and its 68 and 95 per cent regions should enclose the NUTS draws.

In [ ]:
def to_base(name, value):
    prior = priors[name]
    return ndtri((value - float(prior.low)) / (float(prior.high) - float(prior.low)))


def base_jacobian(name, value):
    # d base / d theta for the PreconditionnedUniform prior: 1 / ((high - low) phi(base))
    prior = priors[name]
    base = to_base(name, value)
    return 1.0 / ((float(prior.high) - float(prior.low)) * np.exp(-0.5 * base**2) / np.sqrt(2 * np.pi))


if RUN_SAMPLING:
    potential_jit = jax.jit(potential)
    axes_values = {}
    for name in ("Omega_c", "sigma8"):
        draws = np.asarray(nuts.cosmo[name])
        half_width = max(5 * draws.std(), 0.01)
        axes_values[name] = np.linspace(draws.mean() - half_width, draws.mean() + half_width, GRID_POINTS)
    grid_potential = np.array(
        [
            [
                float(potential_jit({"Omega_c_base": to_base("Omega_c", om), "sigma8_base": to_base("sigma8", s8)}))
                for s8 in axes_values["sigma8"]
            ]
            for om in axes_values["Omega_c"]
        ]
    )
    np.savez(FIXED / "grid.npz", Omega_c=axes_values["Omega_c"], sigma8=axes_values["sigma8"], potential=grid_potential)

grid = np.load(FIXED / "grid.npz")
density = np.exp(-(grid["potential"] - grid["potential"].min())) * np.outer(
    base_jacobian("Omega_c", grid["Omega_c"]), base_jacobian("sigma8", grid["sigma8"])
)
density /= density.sum()
flat = np.sort(density.ravel())[::-1]
levels = [flat[np.searchsorted(np.cumsum(flat), mass)] for mass in (0.95, 0.68)]
for name, axis, marginal in (("Omega_c", grid["Omega_c"], density.sum(1)), ("sigma8", grid["sigma8"], density.sum(0))):
    mean = (marginal * axis).sum()
    print(f"grid: {name:8s} {mean:.4f} +/- {np.sqrt((marginal * (axis - mean) ** 2).sum()):.4f}")

## 9. Posterior of the cosmology

The NUTS draws at fixed IC against the joint draws of notebook 16, with the truth marked. On the right, the grid contours and the NUTS draws on the same axes.

In [ ]:
jfli.infer.plot_posterior([nuts, joint], labels={"Omega_c": r"\Omega_c", "sigma8": r"\sigma_8"})
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.contourf(
    grid["Omega_c"],
    grid["sigma8"],
    density.T,
    levels=[levels[0], levels[1], density.max()],
    colors=["C0", "C0"],
    alpha=0.35,
)
ax.contour(grid["Omega_c"], grid["sigma8"], density.T, levels=levels, colors="C0", linewidths=1)
ax.plot(
    np.asarray(nuts.cosmo["Omega_c"]).ravel(),
    np.asarray(nuts.cosmo["sigma8"]).ravel(),
    ".",
    color="C3",
    ms=4,
    label=nuts.name,
)
ax.plot(
    np.asarray(joint.cosmo["Omega_c"]).ravel(),
    np.asarray(joint.cosmo["sigma8"]).ravel(),
    ".",
    color="C1",
    ms=3,
    label=joint.name,
)
ax.plot(float(cosmo.Omega_c), float(cosmo.sigma8), "k*", ms=14, label="truth")
ax.set(xlabel="$\\Omega_c$", ylabel="$\\sigma_8$", title=f"68 and 95% grid regions, IC fixed ({IC})")
ax.legend(fontsize=8)
plt.show()